In [6]:
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

rng = np.random.default_rng(42)

# -----------------------------
# 1. Basic settings
# -----------------------------
N_USERS = 80
N_TX = 10_000

categories = [
    "Groceries",
    "DiningOut",
    "Transport",
    "Shopping",
    "Entertainment",
    "Utilities",
    "Healthcare",
    "Education",
    "Travel",
    "Bills",
    "PersonalCare",
    "Miscellaneous",
]

# Roughly realistic frequency for each category
cat_probs = np.array([
    0.18,  # Groceries
    0.14,  # DiningOut
    0.10,  # Transport
    0.12,  # Shopping
    0.08,  # Entertainment
    0.07,  # Utilities
    0.05,  # Healthcare
    0.03,  # Education
    0.05,  # Travel
    0.06,  # Bills
    0.05,  # PersonalCare
    0.07,  # Miscellaneous
])
cat_probs = cat_probs / cat_probs.sum()

# -----------------------------
# 2. Vendor list (base + synthetic)
# -----------------------------
vendors_by_category = {
    "Groceries": [
        "Walmart", "Target", "Kroger", "Costco", "Aldi", "Meijer",
        "Whole Foods", "Trader Joes", "Safeway", "Publix",
    ],
    "DiningOut": [
        "McDonalds", "Starbucks", "Subway", "Chipotle", "Dominos",
        "Pizza Hut", "KFC", "Panera Bread", "Olive Garden", "Chick-fil-A",
    ],
    "Transport": [
        "Shell", "BP", "Exxon", "Chevron", "Speedway",
        "Uber", "Lyft", "Metro Transit", "Amtrak", "City Parking Garage",
    ],
    "Shopping": [
        "Amazon", "Best Buy", "IKEA", "Home Depot", "Lowes",
        "Walmart.com", "Target.com", "Apple Store", "Nike", "Adidas",
    ],
    "Entertainment": [
        "Netflix", "Spotify", "Hulu", "DisneyPlus", "AMC Theatres",
        "Regal Cinemas", "Steam", "Xbox Live", "PlayStation Store", "Dave & Busters",
    ],
    "Utilities": [
        "DTE Energy", "Consumers Energy", "Comcast", "AT&T", "Verizon",
        "Spectrum", "Xfinity", "Water Utility", "Gas Utility", "Municipal Utilities",
    ],
    "Healthcare": [
        "CVS Pharmacy", "Walgreens", "Rite Aid", "BlueCross",
        "Urgent Care Clinic", "Family Doctor", "Dental Clinic",
        "Optometry Center", "Physical Therapy", "Local Hospital",
    ],
    "Education": [
        "Udemy", "Coursera", "Khan Academy", "UM-Flint",
        "Community College", "Online Course Platform", "Skillshare",
        "Bookstore Campus", "Language School", "Coding Bootcamp",
    ],
    "Travel": [
        "Delta Airlines", "United Airlines", "American Airlines",
        "Southwest", "Airbnb", "Booking.com", "Hilton Hotels",
        "Marriott Hotels", "Lyft Airport", "Uber Airport",
    ],
    "Bills": [
        "Chase Credit Card", "Capital One Card", "Discover Card",
        "Auto Loan Servicer", "Mortgage Servicer", "Student Loan Servicer",
        "City Taxes", "Property Taxes", "Phone Bill AutoPay", "Internet Bill AutoPay",
    ],
    "PersonalCare": [
        "Ulta Beauty", "Sephora", "Hair Salon", "Barber Shop",
        "Nail Spa", "Massage Envy", "GNC", "Vitamin Shoppe",
        "Gym Membership", "Yoga Studio",
    ],
    "Miscellaneous": [
        "Etsy", "eBay", "Local Gift Shop", "Charity Donation",
        "Thrift Store", "Farmer Market", "Craft Store",
        "Pet Store", "Toy Store", "Office Supply Store",
    ],
}

# Turn into (vendor, primary_category) pairs
vendor_pairs = []
for cat, vs in vendors_by_category.items():
    for v in vs:
        vendor_pairs.append((v, cat))

# Add extra synthetic vendors to reach ~300+
for i in range(1, 221):  # 220 extra
    cat = rng.choice(categories)
    name = f"{cat} Store {i:03d}"
    vendor_pairs.append((name, cat))

print(f"Total unique vendors: {len(vendor_pairs)}")

# -----------------------------
# 3. Helper functions
# -----------------------------
def sample_amount(category: str) -> float:
    """Rough, category-specific amount distributions."""
    if category == "Groceries":
        base = rng.normal(80, 40)
    elif category == "DiningOut":
        base = rng.normal(30, 20)
    elif category == "Transport":
        base = rng.normal(45, 25)
    elif category == "Shopping":
        base = rng.normal(90, 60)
    elif category == "Entertainment":
        base = rng.normal(50, 30)
    elif category == "Utilities":
        base = rng.normal(120, 50)
    elif category == "Healthcare":
        base = rng.normal(150, 80)
    elif category == "Education":
        base = rng.normal(200, 120)
    elif category == "Travel":
        base = rng.normal(250, 150)
    elif category == "Bills":
        base = rng.normal(180, 90)
    elif category == "PersonalCare":
        base = rng.normal(40, 25)
    else:  # Misc
        base = rng.normal(35, 30)

    # Ensure positive + sensible cap
    base = max(5, base)
    base = min(base, 2000)
    # Round to cents
    return round(float(base), 2)


def build_description(vendor: str, category: str, amount: float) -> str:
    templates = [
        "Card purchase at {vendor} for {amount:.2f} USD ({category})",
        "POS transaction - {vendor} ({category}), amount {amount:.2f}",
        "{category} spend at {vendor}, total {amount:.2f}",
        "Online payment to {vendor} - {category}, {amount:.2f}",
        "Mobile app payment at {vendor}, {category}, {amount:.2f}",
    ]
    template = rng.choice(templates)
    return template.format(vendor=vendor, category=category, amount=amount)


payment_methods = ["Debit Card", "Credit Card", "ACH", "Cash", "Digital Wallet"]
cities = ["Detroit", "Ann Arbor", "Chicago", "New York", "San Francisco",
          "Seattle", "Austin", "Miami", "Boston", "Denver"]
states = ["MI", "MI", "IL", "NY", "CA", "WA", "TX", "FL", "MA", "CO"]

date_range = pd.date_range("2023-01-01", "2024-12-31", freq="D")


# -----------------------------
# 4. Generate transactions
# -----------------------------
rows = []

for tx_id in range(1, N_TX + 1):
    user_id = int(rng.integers(1, N_USERS + 1))

    # Choose category according to global frequencies
    category = rng.choice(categories, p=cat_probs)

    # Pick vendor: 85% chance same-category, 15% chance cross-category (noise)
    if rng.random() < 0.85:
        candidates = [v for (v, c) in vendor_pairs if c == category]
    else:
        candidates = [v for (v, c) in vendor_pairs]

    vendor = rng.choice(candidates)

    amount = sample_amount(category)
    desc = build_description(vendor, category, amount)

    date = rng.choice(date_range)
    city_idx = int(rng.integers(0, len(cities)))
    city = cities[city_idx]
    state = states[city_idx]
    payment_method = rng.choice(payment_methods)

    rows.append(
        {
            "transaction_id": tx_id,
            "user_id": user_id,
            "date": date,
            "vendor": vendor,
            "description": desc,
            "amount": amount,
            "payment_method": payment_method,
            "city": city,
            "state": state,
            "category": category,  # LABEL
        }
    )

df_tx = pd.DataFrame(rows)

print(df_tx.shape)
df_tx.head()

Total unique vendors: 340
(10000, 10)


,transaction_id,user_id,date,vendor,description,amount,payment_method,city,state,category
0,1,11,2023-05-10,DiningOut Store 041,"DiningOut spend at DiningOut Store 041, total ...",34.75,Digital Wallet,San Francisco,CA,DiningOut
1,2,2,2024-03-02,Healthcare Store 129,Card purchase at Healthcare Store 129 for 151....,151.75,Cash,Detroit,MI,Healthcare
2,3,51,2023-08-30,BP,Card purchase at BP for 78.98 USD (Transport),78.98,Debit Card,Austin,TX,Transport
3,4,55,2024-02-29,Hulu,"Mobile app payment at Hulu, Entertainment, 22.20",22.20,Credit Card,San Francisco,CA,Entertainment
4,5,49,2024-07-26,Miscellaneous Store 081,"Groceries spend at Miscellaneous Store 081, to...",39.38,Debit Card,Detroit,MI,Groceries


In [8]:
raw_dir = Path("../data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

synthetic_path = raw_dir / "transactions_synthetic.csv"

print("Saved synthetic dataset to:", synthetic_path)

Saved synthetic dataset to: ../data/raw/transactions_synthetic.csv
